In [1]:
# 1 — Setup and raw-file check
from pathlib import Path
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
BASE = next((p for p in candidates if (p / "data" / "processed").exists()), None)
if BASE is None:
    raise FileNotFoundError(f"data/processed not found from {Path.cwd()} - run from inside the project folder.")
RAW = BASE / "data" / "raw"

FILES = {y: {"calls": RAW / f"antaq_{y}_atracacao.txt",
             "cargo": RAW / f"antaq_{y}_carga.txt",
             "cargo_items": RAW / f"antaq_{y}_carga_conteinerizada.txt"} for y in (2024, 2025)}

for y, d in FILES.items():
    for k, p in d.items():
        status = f"OK  {p.stat().st_size/1e6:8.1f} MB" if p.exists() else "MISSING"
        print(f"{y} {k:12s} {status}")

2024 calls        OK      36.8 MB
2024 cargo        OK     451.1 MB
2024 cargo_items  OK     294.0 MB
2025 calls        OK      40.3 MB
2025 cargo        OK     453.5 MB
2025 cargo_items  OK     272.4 MB


In [2]:
# 2 — Read calls and container cargo for both years with the same code
CALL_COLS = {"IDAtracacao": "call_id", "Terminal": "terminal_raw", "Porto Atracação": "port",
             "Tipo de Navegação da Atracação": "navigation", "Nº do IMO": "imo",
             "Data Chegada": "ts_arrival", "Data Atracação": "ts_berth", "Data Desatracação": "ts_unberth"}
CARGO_COLS = ["IDAtracacao", "Origem", "Destino", "CDMercadoria", "Tipo Operação da Carga",
              "ConteinerEstado", "FlagMCOperacaoCarga", "Sentido", "TEU"]

def read_year(y):
    calls = pd.read_csv(FILES[y]["calls"], sep=";", encoding="utf-8-sig", dtype=str,
                        usecols=list(CALL_COLS)).rename(columns=CALL_COLS)
    calls["call_id"] = calls["call_id"].astype(int)
    for c in ["ts_arrival", "ts_berth", "ts_unberth"]:
        calls[c] = pd.to_datetime(calls[c], format="%d/%m/%Y %H:%M:%S", errors="coerce")
    calls["year"] = y

    cargo = pd.read_csv(FILES[y]["cargo"], sep=";", encoding="utf-8-sig", dtype=str, usecols=CARGO_COLS)
    cargo["TEU"] = pd.to_numeric(cargo["TEU"].str.replace(",", "."), errors="coerce")
    cargo = cargo[(cargo["TEU"] > 0) & (cargo["FlagMCOperacaoCarga"] == "1")].copy()
    cargo = cargo.rename(columns={"IDAtracacao": "call_id", "Origem": "origin", "Destino": "dest",
                                  "CDMercadoria": "iso_code", "Tipo Operação da Carga": "op_type",
                                  "ConteinerEstado": "state", "Sentido": "direction", "TEU": "teu"})
    cargo["call_id"] = cargo["call_id"].astype(int)
    cargo["equip"] = cargo["iso_code"].str[2].map({"G": "dry", "R": "reefer"}).fillna("other")
    cargo["orig_cc"] = cargo["origin"].str[:2]
    cargo["dest_cc"] = cargo["dest"].str[:2]
    cargo["trade"] = cargo["op_type"].str.startswith("Longo Curso") & \
                     ~((cargo["orig_cc"] == "BR") & (cargo["dest_cc"] == "BR"))
    cargo["year"] = y
    return calls, cargo.drop(columns=["FlagMCOperacaoCarga"])

data = {y: read_year(y) for y in (2024, 2025)}
for y, (calls, cargo) in data.items():
    orphan = ~cargo["call_id"].isin(calls["call_id"])
    print(f"{y}: {len(calls):,} calls | {len(cargo):,} container rows | {cargo['teu'].sum():,.0f} TEU | "
          f"{cargo['call_id'].nunique():,} calls with containers | cargo rows without a call: {orphan.sum()}")

2024: 106,834 calls | 1,748,623 container rows | 13,907,871 TEU | 9,456 calls with containers | cargo rows without a call: 0
2025: 116,327 calls | 1,756,613 container rows | 15,344,570 TEU | 10,084 calls with containers | cargo rows without a call: 0


In [3]:
# 3 — Validate 2025 against the project-2 panel base, call by call
calls25, cargo25 = data[2025]
panel = pd.read_csv(BASE / "data" / "processed" / "calls_2025.csv")

bulk_teu = cargo25.groupby("call_id")["teu"].sum()
panel_teu = panel.set_index("call_id")["teu"]

both = bulk_teu.index.intersection(panel_teu.index)
only_bulk = bulk_teu.index.difference(panel_teu.index)
only_panel = panel_teu.index.difference(bulk_teu.index)
print(f"Calls in both: {len(both):,} | only in bulk: {len(only_bulk):,} | only in panel: {len(only_panel):,}")

gap = (bulk_teu[both] - panel_teu[both]).abs()
print(f"Common calls with TEU gap > 1: {(gap > 1).sum():,} | total abs gap: {gap.sum():,.0f} TEU")
print(f"TEU in bulk-only calls: {bulk_teu[only_bulk].sum():,.0f} | TEU in panel-only calls: {panel_teu[only_panel].sum():,.0f}\n")

look = calls25.set_index("call_id")
for label, ids in [("Only in bulk", only_bulk), ("Only in panel", only_panel)]:
    if len(ids):
        sub = look.reindex(ids)
        print(f"{label} - by navigation:", sub["navigation"].value_counts(dropna=False).to_dict())
        print(f"{label} - by unberth month:", sub["ts_unberth"].dt.month.value_counts().sort_index().to_dict())
        print(f"{label} - top terminals:", sub["terminal_raw"].value_counts().head(5).to_dict(), "\n")

Calls in both: 9,999 | only in bulk: 85 | only in panel: 8
Common calls with TEU gap > 1: 0 | total abs gap: 0 TEU
TEU in bulk-only calls: 113,032 | TEU in panel-only calls: 1,269

Only in bulk - by navigation: {'Longo Curso': 80, 'Cabotagem': 5}
Only in bulk - by unberth month: {2: 85}
Only in bulk - top terminals: {'TCP': 68, 'CAIS PÚBLICO': 17} 

Only in panel - by navigation: {nan: 7, 'Interior': 1}
Only in panel - by unberth month: {12.0: 1}
Only in panel - top terminals: {'Porto Chibatão': 1} 



In [4]:
# 4 — Year-on-year view: deep-sea trade by direction, state and equipment (TEU, thousands)
trade = pd.concat([cargo[cargo["trade"]] for _, cargo in data.values()], ignore_index=True)
trade["direction"] = trade["direction"].map({"Embarcados": "Export", "Desembarcados": "Import"})
trade["state"] = trade["state"].map({"Cheio": "Full", "Vazio": "Empty"})

yoy = (trade.groupby(["direction", "equip", "state", "year"])["teu"].sum() / 1e3).unstack("year").round(0)
yoy["change_%"] = (100 * (yoy[2025] / yoy[2024] - 1)).round(1)
print(yoy.to_string(), "\n")

tot = trade.groupby("year")["teu"].sum() / 1e3
print(f"Deep-sea trade TEU (thousands): 2024 {tot[2024]:,.0f} | 2025 {tot[2025]:,.0f} | "
      f"change {100*(tot[2025]/tot[2024]-1):.1f}%")

dry_empty_exp = trade[(trade["direction"] == "Export") & (trade["equip"] == "dry") & (trade["state"] == "Empty")]
by_dest = (dry_empty_exp.groupby(["dest_cc", "year"])["teu"].sum() / 1e3).unstack("year").fillna(0).round(0)
by_dest["change"] = by_dest[2025] - by_dest[2024]
print("\nDry empty exports by destination country (TEU, thousands):")
print(by_dest.sort_values(2025, ascending=False).head(10).to_string())

year                      2024    2025  change_%
direction equip  state                          
Export    dry    Empty   847.0  1258.0      48.5
                 Full   2257.0  2334.0       3.4
          other  Empty    46.0    61.0      32.6
                 Full     52.0    52.0       0.0
          reefer Empty    35.0    36.0       2.9
                 Full    800.0   852.0       6.5
Import    dry    Empty   552.0   576.0       4.3
                 Full   3011.0  3359.0      11.6
          other  Empty    12.0    11.0      -8.3
                 Full     83.0    81.0      -2.4
          reefer Empty   360.0   427.0      18.6
                 Full    330.0   296.0     -10.3 

Deep-sea trade TEU (thousands): 2024 8,386 | 2025 9,342 | change 11.4%

Dry empty exports by destination country (TEU, thousands):
year      2024   2025  change
dest_cc                      
CN       391.0  696.0   305.0
SG       166.0  176.0    10.0
XZ        14.0   74.0    60.0
PA        63.0   61.0    -2.0
H

In [5]:
# 5 — Save the two-year base for the next notebooks
calls_all = pd.concat([calls[calls["call_id"].isin(cargo["call_id"].unique())]
                       for calls, cargo in data.values()], ignore_index=True)
teu_call = pd.concat([cargo for _, cargo in data.values()]).groupby("call_id")["teu"].sum().rename("teu")
calls_all = calls_all.merge(teu_call, on="call_id", validate="1:1")
calls_all.to_csv(BASE / "data" / "processed" / "calls_2024_2025.csv", index=False)

trade["foreign_port"] = np.where(trade["direction"] == "Export", trade["dest"], trade["origin"])
trade["foreign_cc"] = trade["foreign_port"].str[:2]
flows = (trade.groupby(["year", "call_id", "direction", "state", "equip", "foreign_port", "foreign_cc"],
                       as_index=False)["teu"].sum())
flows.to_csv(BASE / "data" / "processed" / "trade_flows_2024_2025.csv.gz", index=False, compression="gzip")

print(f"calls_2024_2025.csv: {len(calls_all):,} calls "
      f"({(calls_all['year']==2024).sum():,} in 2024, {(calls_all['year']==2025).sum():,} in 2025)")
print(f"trade_flows_2024_2025.csv.gz: {len(flows):,} rows, "
      f"{(BASE/'data'/'processed'/'trade_flows_2024_2025.csv.gz').stat().st_size/1e6:.1f} MB")

calls_2024_2025.csv: 19,540 calls (9,456 in 2024, 10,084 in 2025)
trade_flows_2024_2025.csv.gz: 280,449 rows, 1.5 MB


In [6]:
# 6 — Two checks before the analysis: the unknown 'XZ' code and the net dry-box balance
xz = flows[flows["foreign_cc"] == "XZ"]
print("XZ by foreign port code (TEU, thousands):")
print((xz.groupby(["foreign_port", "year"])["teu"].sum() / 1e3).unstack("year").fillna(0).round(1)
      .sort_values(2025, ascending=False).head(10).to_string(), "\n")

xz_br = xz.merge(calls_all[["call_id", "port"]], on="call_id", validate="m:1")
print("XZ by Brazilian port of call (TEU, thousands):")
print((xz_br.groupby(["port", "year"])["teu"].sum() / 1e3).unstack("year").fillna(0).round(1)
      .sort_values(2025, ascending=False).head(10).to_string(), "\n")

dry = (flows[flows["equip"] == "dry"].groupby(["year", "direction"])["teu"].sum() / 1e3).unstack()
dry["net_inflow"] = dry["Import"] - dry["Export"]
print("Dry boxes, full + empty (TEU, thousands):")
print(dry.round(0).to_string())

XZ by foreign port code (TEU, thousands):
year          2024  2025
foreign_port            
XZZZZ         16.8  80.4 

XZ by Brazilian port of call (TEU, thousands):
year                                  2024  2025
port                                            
Super Terminais Comércio e Indústria  16.8  80.4
Vila do Conde                          0.0   0.0 

Dry boxes, full + empty (TEU, thousands):
direction  Export  Import  net_inflow
year                                 
2024       3104.0  3563.0       459.0
2025       3592.0  3935.0       343.0


## What this notebook establishes

- **One source for both years.** ANTAQ's bulk download (extracted 17 July 2026) is used for 2024 and 2025 with the same code. Only cargo flagged as port throughput (`FlagMCOperacaoCarga = 1`) is kept, which excludes restows.
- **2025 matches the panel used in the main notebook.** 9,999 container calls are in both with an identical TEU figure per call. The bulk file adds 85 calls, all at Paranaguá in February (113k TEU), the month the panel export was missing. Eight later-added calls (1.3k TEU) are only in the panel.
- **Container type comes from ISO size-type codes** (third character: G = dry, R = reefer). Ventilated units are practically zero, which confirms that the panel's "Ventilado High Cube" label is a recording convention, not a cargo type.
- **Deep-sea trade grew 11.4%** (8.39 → 9.34 million TEU, Brazil-to-Brazil pairs excluded).
- **Dry empty exports grew 48.5%** (847k → 1,258k TEU), far faster than full dry imports (+11.6%). China accounts for 305k of the increase. About 60k go to an unidentified foreign port (`XZZZZ`), almost all from one Manaus terminal.
- **The maritime balance does not close.** Around 350–460k more dry TEU enter than leave by deep-sea each year. They most likely leave through channels this data does not cover (overland transit to neighbouring countries, excluded transhipment categories, disposals), so this study makes no claim about container stock building up in Brazil.